# TPU Smoke Test
Verify TPU quota, torch_xla, device access, and model loading.

In [ ]:
import os, sys, time
print('=== Step 1: Check environment ===')
print(f'Python: {sys.version}')

# Check if TPU env vars are set
tpu_name = os.environ.get('TPU_NAME', 'NOT SET')
xrt_workers = os.environ.get('XRT_TPU_CONFIG', 'NOT SET')
print(f'TPU_NAME: {tpu_name}')
print(f'XRT_TPU_CONFIG: {xrt_workers}')

print('\n=== Step 2: Import torch_xla ===')
import torch
print(f'PyTorch: {torch.__version__}')

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f'torch_xla: {torch_xla.__version__}')
    XLA_OK = True
except ImportError as e:
    print(f'torch_xla import FAILED: {e}')
    XLA_OK = False

if XLA_OK:
    print('\n=== Step 3: Get TPU device ===')
    device = xm.xla_device()
    print(f'TPU device: {device}')
    print(f'Device type: {device.type}')

    print('\n=== Step 4: Tensor operations ===')
    t = torch.randn(2, 3).to(device)
    print(f'Tensor on TPU: {t}')
    result = t @ t.T
    print(f'MatMul result: {result}')

    print('\n=== Step 5: Larger operation (Conv2d) ===')
    conv = torch.nn.Conv2d(3, 64, 3, padding=1).to(device)
    x = torch.randn(1, 3, 96, 96).to(device)
    start = time.time()
    y = conv(x)
    xm.mark_step()  # Force XLA execution
    print(f'Conv2d output: {y.shape}, took {time.time()-start:.3f}s')

    print('\n=== Step 6: Load VSR model checkpoint ===')
    import glob
    MODEL_PATH = None
    for c in glob.glob('/kaggle/input/**/vsr_model.pth', recursive=True):
        if os.path.getsize(c) > 1e6:
            MODEL_PATH = c
            break
    if MODEL_PATH:
        print(f'Model found: {MODEL_PATH} ({os.path.getsize(MODEL_PATH)/1e6:.0f} MB)')
        ckpt = torch.load(MODEL_PATH, map_location='cpu')
        print(f'Checkpoint keys: {len(ckpt)} layers')
        # Check a layer's shape
        first_key = list(ckpt.keys())[0]
        print(f'First layer: {first_key} shape={ckpt[first_key].shape}')
        # Try moving a weight to TPU
        w = ckpt[first_key].to(device)
        print(f'Weight on TPU: {w.device}, shape={w.shape}')
        del ckpt, w
        print('Model checkpoint loads and transfers to TPU OK')
    else:
        print('Model NOT FOUND')

    print('\n=== Step 7: CTC loss test ===')
    try:
        T, C, N, S = 50, 100, 1, 10
        log_probs = torch.randn(T, N, C, device=device).log_softmax(2)
        targets = torch.randint(1, C, (N, S), dtype=torch.long, device=device)
        input_lengths = torch.full((N,), T, dtype=torch.long, device=device)
        target_lengths = torch.full((N,), S, dtype=torch.long, device=device)
        loss = torch.nn.functional.ctc_loss(log_probs, targets, input_lengths, target_lengths, blank=0, reduction='none')
        xm.mark_step()
        print(f'CTC loss on TPU: {loss.item():.4f} — OK!')
    except Exception as e:
        print(f'CTC loss on TPU FAILED: {e}')
        print('Will need CPU fallback for CTC scoring')

    print('\n=== RESULT: TPU OK! ===')
else:
    print('\n=== RESULT: TPU NOT AVAILABLE ===')